In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:

from wringlet import (
    data_provenance_enabled, 
    data_provenance_session_builder,
    provenance_column_name
)

In [3]:
from pyspark.sql import SparkSession

spark = (
    # SparkSession
    # .builder
    data_provenance_session_builder()
    .appName("data-provenance-notebook")
    .getOrCreate()
)

26/07/16 14:27:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### Create toy dataframe

In [4]:
from datetime import date
import pandas as pd

df = spark.createDataFrame([
    ("A", date(2026, 1, 15), 10.0, 90),
    ("A", date(2026, 1, 16), 10.0, 120),
    ("A", date(2026, 1, 17), 5.0, 300),
    ("B", date(2026, 1, 15), 100.0, 20),
    ("B", date(2026, 1, 16), 100.0, 30),
    ("B", date(2026, 1, 17), 80.0, 60),
], ["product", "date", "price", "sales"]
)
df.printSchema()

df.toPandas()

root
 |-- product: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- sales: long (nullable = true)



,product,date,price,sales
0,A,2026-01-15,10.0,90
1,A,2026-01-16,10.0,120
2,A,2026-01-17,5.0,300
3,B,2026-01-15,100.0,20
4,B,2026-01-16,100.0,30
5,B,2026-01-17,80.0,60


### Test with pyspark syntax

In [5]:
df2 = df.select("product")

df2.toPandas()

,product
0,A
1,A
2,A
3,B
4,B
5,B


In [6]:
with data_provenance_enabled(spark, df) as df_with_provenance:
    res = df_with_provenance.toPandas()

res

,product,date,price,sales,_provenance_tag
0,A,2026-01-15,10.0,90,d9829464-4b66-421f-ab42-6951ed1cdf9e
1,A,2026-01-16,10.0,120,ee206ca1-818f-4d75-8ebc-4d29542da9a2
2,A,2026-01-17,5.0,300,a8c569f2-b1d6-4b44-a32f-0b215d84c2b3
3,B,2026-01-15,100.0,20,1438165a-fc23-48dd-bb5b-4172ec054b33
4,B,2026-01-16,100.0,30,e9a108c4-cdaa-4ebc-8d4e-21a34937ad21
5,B,2026-01-17,80.0,60,8a4412e2-e80a-48db-b4ba-4affe35bcdc0


### Test with SQL syntax

In [7]:
df.createOrReplaceTempView("sales")

spark.sql("select * from sales").toPandas()

,product,date,price,sales
0,A,2026-01-15,10.0,90
1,A,2026-01-16,10.0,120
2,A,2026-01-17,5.0,300
3,B,2026-01-15,100.0,20
4,B,2026-01-16,100.0,30
5,B,2026-01-17,80.0,60


In [8]:
df.createOrReplaceTempView("sales")

with data_provenance_enabled(spark, "sales") as view_with_provenance:
    res = spark.table(view_with_provenance).toPandas()

res

,product,date,price,sales,_provenance_tag
0,A,2026-01-15,10.0,90,8e2b4383-2604-4989-94c5-479fad4bea77
1,A,2026-01-16,10.0,120,34501692-a278-4152-8cf0-6f0ca65f930b
2,A,2026-01-17,5.0,300,e072aef4-e8ab-486e-aa43-fa2e8fd62249
3,B,2026-01-15,100.0,20,16d51bfa-8456-4323-b6e8-6d355805feb9
4,B,2026-01-16,100.0,30,d937bc9f-4c23-43a7-8fbf-023167b722f3
5,B,2026-01-17,80.0,60,895e235e-d9d5-4dce-907c-ac5b10a23354


### Test with multiple dataframes/views and with custom name for provenance column

In [9]:
spark.conf.set("spark.provenance.columnName", "toto")
print(provenance_column_name(spark))

with data_provenance_enabled(spark, df, "sales", df2) as (df_with_provenance, view_with_provenance, df2_with_provenance):
    res = df_with_provenance.toPandas()
    res2 = spark.table(view_with_provenance).toPandas()
    res3 = df2_with_provenance.toPandas()

print(res)
print(res2)
res3


toto
  product        date  price  sales                                  toto
0       A  2026-01-15   10.0     90  93db65da-a2a3-453a-95b8-777eb9062ec9
1       A  2026-01-16   10.0    120  c081c743-2f8e-432c-809b-14245de6a2f7
2       A  2026-01-17    5.0    300  b610e135-bf57-4030-81cb-d3e99f36254a
3       B  2026-01-15  100.0     20  2a514aea-0ae7-45e1-b011-1e6630764210
4       B  2026-01-16  100.0     30  2f4c282f-f762-451a-ab7d-edab86813a18
5       B  2026-01-17   80.0     60  4c07ae0e-3079-43d0-af1d-dce5723815d6
  product        date  price  sales                                  toto
0       A  2026-01-15   10.0     90  5bef35f5-5eaf-4750-bbed-ace4e844fa91
1       A  2026-01-16   10.0    120  47d92bc1-56ed-442d-8e67-8e6ec0397de4
2       A  2026-01-17    5.0    300  345e0206-ca14-4f77-8e1b-b41559741fdb
3       B  2026-01-15  100.0     20  c8f55538-fa8f-4133-9e0a-ba5652ea91ef
4       B  2026-01-16  100.0     30  950dd9cf-0ab3-4bfb-ad63-c74abe2a86fc
5       B  2026-01-17   80.0     

,product,toto
0,A,e7fdbb0f-776d-4bf2-82eb-787604f69e9b
1,A,14ca6f31-a0f5-409f-8bfc-be18658e4327
2,A,f42a0b56-7a18-43fb-b5b5-901022bd7915
3,B,56dc0875-f01e-42a6-868c-3bcf519a4b12
4,B,eb650c27-9559-42d3-a98b-a306ae1bfea4
5,B,99f7a539-c483-41a7-b1ce-c2b2c4ef7fda
